In [6]:
# Bloco 1: Instalação de dependências e Configuração do Ambiente

# Instalação das bibliotecas necessárias
!pip install -q --upgrade diffusers transformers accelerate peft datasets torch torchvision torchaudio torchao>=0.6.0


import torch
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Autenticação no Hugging Face

try:
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("hugging-face")
    login(token)
    print("Autenticação no Hugging Face realizada com sucesso!")
except Exception as e:
    print("Erro na autenticação. Verifique seu Secret no Kaggle ou insira o token manualmente.")

# 2. Configuração de dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de processamento: {device}")

Autenticação no Hugging Face realizada com sucesso!
Dispositivo de processamento: cuda


In [3]:
# Bloco 2: Carregamento dos Modelos (Pipeline Multimodal)

from diffusers import StableDiffusionPipeline
from transformers import pipeline
import torch

print("Carregando modelos... (isso pode levar alguns minutos)")

# 1. Carregar Stable Diffusion com seu LoRA
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
lora_id = "homerio/estilo_arquitetura_modernista_brasilia_v4"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16
).to(device)

pipe.load_lora_weights(lora_id)
print("LoRA carregado com sucesso!")

# 2. Carregar LLM para expansão de prompt (Qwen 0.5B - leve e rápido)
expansor = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=device)

# 3. Carregar TTS (Bark - pequeno e eficiente)
# Nota: O Bark pode baixar modelos adicionais na primeira execução
tts = pipeline("text-to-speech", model="suno/bark-small", device=device)

print("Todos os modelos foram carregados com sucesso!")

Carregando modelos... (isso pode levar alguns minutos)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


LoRA carregado com sucesso!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

Todos os modelos foram carregados com sucesso!


In [ ]:
import gradio as gr

def pipeline_multimodal(tema):
    # 1. LLM expande o tema
    # Nota: Removi o excesso de texto para evitar que o modelo se perca
    instrucao = (f"Crie uma descrição visual detalhada de {tema}, em apenas uma frase, "
                 "inicie a resposta exatamente com: 'estilo_arquitetura_modernista_brasilia'" )
    
    # Gerar prompt: usamos 'do_sample=True' para variar e 'clean_up_tokenization_spaces=False'
    resultado_llm = expansor(
        instrucao, 
        max_new_tokens=60, 
        do_sample=True, 
        temperature=0.7,
        clean_up_tokenization_spaces=False
    )
    
    texto_gerado = resultado_llm[0]["generated_text"]
    
    # 2. LIMPEZA DINÂMICA
    # Procuramos onde a nossa palavra-chave começa e pegamos tudo dali para frente
    keyword = "estilo_arquitetura_modernista_brasilia"
    if keyword in texto_gerado:
        prompt_final = texto_gerado[texto_gerado.find(keyword):].strip()
    else:
        # Fallback: caso o modelo "esqueça" a instrução, usamos o prompt básico
        prompt_final = f"{keyword}, {tema}"
    
    # 3. Stable Diffusion gera a imagem
    imagem = pipe(prompt,  negative_prompt="desfocado, deformado, sobreposição de elementos, elemetos desconexos", 
                  num_inference_steps=30, guidance_scale=7.5).images[0]
    
    # 4. TTS gera o áudio
    audio_data = tts(prompt)
    
    return prompt, imagem, (audio_data["sampling_rate"], audio_data["audio"])

# 5. Interface Gradio
demo = gr.Interface(
    fn=pipeline_multimodal,
    inputs=gr.Textbox(label="Tema", placeholder="Ex: Catedral Metropolitana"),
    outputs=[
        gr.Textbox(label="Prompt Final"),
        gr.Image(label="Imagem"),
        gr.Audio(label="Descrição")
    ],
    title="Ateliê Generativo - Homerio Junior: Arquitetura de Brasília"
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://fd486a0bb8e13cb7d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
